In [69]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
import os 
from dotenv import load_dotenv
import json
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL"),
    temperature=0
)

In [33]:
from langchain.tools import tool

@tool
def tool_duckduckgo_search(query: str) -> str: 
    """
    Use this tool to search for current/latest information on the web using DuckDuckGo.
    This function takes a query string as input and returns the search results.
    """
    from langchain_community.tools import DuckDuckGoSearchResults

    tool = DuckDuckGoSearchResults()
    return tool.invoke(query)

In [21]:
tool_duckduckgo_search.invoke("What is the latest news on Olaelectric?")

'snippet: We provide you with the latest news on Finance and Finance tips straight from the experts. ... The content we generate is available on desktop ..., title: olaelectric - Business League, link: https://www.businessleague.in/tag/olaelectric/, snippet: We publish the latest EV news, insightful features in our digital magazine, and foster a passionate community of EV enthusiasts, innovators, and ..., title: OlaElectric Archives -, link: https://humansofev.info/tag/olaelectric/, snippet: #olaelectric and its future in India - ZigAnalysis ... NEWS & REVIEWS ... Latest Cars ... Latest, title: #olaelectric and its future in India - ZigAnalysis, link: https://www.zigwheels.com/gallery/reviews/olaelectric-and-its-future-in-india-ziganalysis/51191/1, snippet: Ola Electric Charges Ahead: Securing Debt Funding and Revving Up for Growth The electric mobility revolution continues to gain momentum,, title: #OlaElectric #ElectricVehicles, link: https://startupnewswire.in/tag/olaelectric-electr

In [34]:
@tool
def tool_wikipedia_search(query: str) -> str:

    """
    Use this tool to search for information about famous people, places, and things on Wikipedia.
    This function takes a query string as input and returns the search results.
    """
    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wikipedia.invoke(query)

In [35]:
@tool
def tool_personal_info(name: str) -> str:
    """
    Use this tool when you need to answer questions about personal information.
    Args: name (str): The name of the person you want to get information about.
    Returns: str: A string containing the personal information of the person.
    """
    infos = [
        {
            "name": "John Doe",
            "age": 30,
            "occupation": "Software Engineer"
        },
        {
            "name": "Jane Smith",
            "age": 25,
            "occupation": "Data Scientist"
        }
    ]
    for info in infos:
        if info["name"].lower() in name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [36]:
tool_personal_info.invoke("Jane Smith")

'Jane Smith is 25 years old and works as a Data Scientist.'

## Bind tools : it does not execute the tools. It only gives the LLM permission to decide which tool should be called.

In [37]:
toolkit = [tool_duckduckgo_search, tool_wikipedia_search, tool_personal_info]

In [38]:
llm_bind = llm.bind_tools(toolkit)

in above cell, llm says that it thinks that tool_personal_info should be called.

In [42]:
responsee = llm_bind.invoke('tell me about john doe')

In [43]:
print(responsee)

content=[] additional_kwargs={'function_call': {'name': 'tool_personal_info', 'arguments': '{"name": "John Doe"}'}, '__gemini_function_call_thought_signatures__': {'svy6pc4z': 'EpMECpAEAQw51seJNKyfkmkrbbGi664ZE58jiGcEF+AljJNipgQHMF1dA9YOOVsyCSqcfkL0pjUgLBimcPBDwn/M8q7M8I3f+74sHIEbYsEgDahrqoG4r4+Yz3pA1zTFA26Y89efeDTRP/VTGqjc5tSMi5ZLareSn0ZRe3CB22QD+DITpjaCN+ogViLm7tFeUJ7Mgj+hVBtFAXLnwYAqz2J1QEkefj1Aex8xYHMI+LxYa4gBP988mgzQ97tYvR5cwWogfgU+CIp+Fihdm2kNshLZeO8Gn+WaE0ZKJwGMmeEWdZjNY5yn+lJOZt8ykgp+lCLBYdHka+WIzku9eq3pS/oBiQNsZxYz7qHK0mlNHvVHoiZt4ILiO8J7Nvzs/uwN9X8vsl/YTTdStYvhAf/swXoLv3vlm1zsR1bDvIAMVWVOFUsBcucBhlIx3e2LxdPRGAprTwsZ0mIXOwHcdiwzCHShAbfwYbTOR2ZRGOtM7juRCcd5JSGdDn86blsNegmku8XsaGGxBFf2bfemtFNpBlPmHyh7ULIfkw3FesH/CodFw2EnAaH73eNLj0F2Wd58mVuM0GjHL8eGWgeTwUk9JHEPbb49g0QeHJlSTsO1Nr4UtqxCzp7EMLYbk4doQeV974POvQ5TdQ8+3/AtrLdB59oEC/H25rqZK4yCvC8l+KfXRaw1Wm/DtVM6UIp+8U5QvVkLCdDP'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], '

In [73]:
tool_call = responsee.additional_kwargs['function_call']
tool_name = tool_call['name']
print(f"Tool called: {tool_name}")
tool_arguments = tool_call['arguments'] # this is a string. convert into dictionary using json.loads
tool_arguments = json.loads(tool_arguments)
tool_arguments = tool_arguments['name'].strip()
print(f"Arguments passed to the tool: {tool_arguments}")

Tool called: tool_personal_info
Arguments passed to the tool: John Doe


## create REACT agent

In [75]:
from langchain.agents import create_agent

In [76]:
my_agent = create_agent(llm, toolkit)

In [ ]:
my_agent.invoke("tell me about john doe")